In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import os 
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity



In [2]:
os.makedirs("output", exist_ok=True)

In [3]:
customer_df = pd.read_csv("C:/Users/Javaughn/Documents/GitHub/ecommerce-data-visualization-project/synthetic_online_retail_data.csv")
 
customer_df.head

customer_df["order_date"] = pd.to_datetime(customer_df["order_date"], errors='coerce')
customer_df["review_score"] = customer_df["review_score"].fillna(0)
customer_df["gender"] = customer_df["gender"].fillna(customer_df["gender"].mode()[0])

customer_df["revenue"] = customer_df["quantity"] * customer_df["price"]

print(f" Data Loaded - {len(customer_df):,}transactions | {customer_df['customer_id'].nunique()} unique customers")
print(f" Date Range: {customer_df['order_date'].min().date()} -> {customer_df['order_date'].max().date()}")
print(f" NaNs remaining: {customer_df.isnull().sum().sum()}")
print(customer_df.head(3).to_string(index=False))

 Data Loaded - 1,000transactions | 1000 unique customers
 Date Range: 2024-03-19 -> 2025-03-19
 NaNs remaining: 0
 customer_id order_date  product_id  category_id     category_name product_name  quantity  price payment_method           city  review_score gender  age  revenue
       13542 2024-12-17         784           10       Electronics   Smartphone         2 373.36    Credit Card New Oliviaberg           1.0      F   56   746.72
       23188 2024-06-01         682           50 Sports & Outdoors  Soccer Ball         5 299.34    Credit Card   Port Matthew           0.0      M   59  1496.70
       55098 2025-02-04         684           50 Sports & Outdoors         Tent         5  23.00    Credit Card     West Sarah           5.0      F   64   115.00


In [4]:
snapshot = customer_df["order_date"].max() + pd.Timedelta(days=1)

In [5]:
rfm = (customer_df.groupby("customer_id")).agg(
        recency = ("order_date", lambda x: (snapshot - x.max()).days), 
        frequency = ("product_id", "nunique"), 
        monetary = ("revenue", "sum"), 
        avg_review = ("review_score", "mean"), 
        avg_qty = ("quantity", "mean"), 
        age = ("age", "first")
    ).reset_index()

print(f" RFM table - {rfm.shape[0]} customers")
print(rfm.describe().round(2).to_string())

 RFM table - 1000 customers
       customer_id  recency  frequency  monetary  avg_review  avg_qty      age
count      1000.00  1000.00     1000.0   1000.00     1000.00  1000.00  1000.00
mean      55490.72   186.21        1.0    737.33        3.19     2.95    46.38
std       25910.19   104.83        0.0    566.40        1.95     1.41    16.57
min       10201.00     1.00        1.0     20.84        0.00     1.00    18.00
25%       33857.00    93.00        1.0    285.84        1.00     2.00    32.00
50%       54619.50   187.00        1.0    592.79        4.00     3.00    47.00
75%       77848.50   277.00        1.0   1081.04        5.00     4.00    61.00
max       99923.00   366.00        1.0   2437.65        5.00     5.00    75.00
